<a href="https://colab.research.google.com/github/alanfel/Proyecto_Integrador/blob/main/Proyecto_Diplomado_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Desactivar logs innecesarios de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, accuracy_score, f1_score
)
from statsmodels.tsa.seasonal import seasonal_decompose
import gradio as gr

# 1. GENERACIÓN DEL DATASET SINTÉTICO ACTIVO (df_master)
np.random.seed(42)
n_records = 600
fechas = pd.date_range(start="1975-01-01", periods=n_records, freq="ME")

# Simulación realista de factores macro-climáticos colombianos (estacionalidad y ciclos de El Niño)
t = np.linspace(0, 10 * np.pi, n_records)
# Ciclo ONI: oscilación lenta (Enso) + ruido estocástico
indice_oni = 1.5 * np.sin(t / 2) + np.random.normal(0, 0.3, n_records)

# Lluvias e hidrología inversamente correlacionadas con ONI
precipitacion = 180 - 45 * indice_oni + np.random.normal(0, 15, n_records)
precipitacion = np.clip(precipitacion, 30, 450)

temperatura = 21 + 1.2 * indice_oni + np.random.normal(0, 0.5, n_records)
temperatura = np.clip(temperatura, 15, 30)

aportes_hidricos = 100 - 25 * indice_oni + np.random.normal(0, 8, n_records)
aportes_hidricos = np.clip(aportes_hidricos, 20, 160)

volumen_embalses = 70 - 18 * indice_oni + np.random.normal(0, 6, n_records)
volumen_embalses = np.clip(volumen_embalses, 15, 99)

demanda_diaria = 120 + 0.15 * np.arange(n_records) + np.random.normal(0, 3, n_records)

# El Precio de Bolsa sube fuertemente cuando ONI es alto (Sequía / El Niño) y los embalses bajan
precio_bolsa = 120 + 150 * np.maximum(0, indice_oni)**1.5 + (100 - volumen_embalses) * 2.5 + np.random.normal(0, 20, n_records)
precio_bolsa = np.clip(precio_bolsa, 50, 950)

# Componente G_CEO de la tarifa de Cauca tiene inercia y tendencia
componente_g = 180 + 1.1 * precio_bolsa * 0.4 + 0.5 * np.arange(n_records) + np.random.normal(0, 15, n_records)
componente_g = np.clip(componente_g, 120, 1200)

df_master = pd.DataFrame({
    'Fecha': fechas,
    'Precio_Bolsa_Nacional_COP_kWh': precio_bolsa,
    'Componente_G_CEO_Cauca_COP_kWh': componente_g,
    'Aportes_Hidricos_Colombia_Pct': aportes_hidricos,
    'Volumen_Util_Embalses_Pct': volumen_embalses,
    'Demanda_Nacional_GWh_Dia': demanda_diaria,
    'Indice_ONI_Clima_Hist': indice_oni,
    'Precipitacion_Promedio_mm': precipitacion,
    'Temperatura_Promedio_C': temperatura
})

# Categorización ONI
def categorizar_clima(oni):
    if oni >= 0.5:
        return 'El Niño (Cálido/Seco)'
    elif oni <= -0.5:
        return 'La Niña (Frío/Lluvioso)'
    else:
        return 'Neutro'

df_master['Estado_Climatico'] = df_master['Indice_ONI_Clima_Hist'].apply(categorizar_clima)
num_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()

print("Dataset consolidado con éxito. Registros:", len(df_master))


Dataset consolidado con éxito. Registros: 600


In [5]:
# 2. INICIALIZACIÓN DE LA UI CON GRADIO (ESTILO Y COMPONENTES)
sns.set_theme(style="whitegrid")

def tab1_get_data(num_rows):
    return df_master.head(int(num_rows))

def tab2_get_stats():
    return df_master[num_cols].describe().reset_index()

def tab3_plot_dist(variable):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_master[variable], kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f'Histograma y KDE de {variable}')
    sns.boxplot(y=df_master[variable], color='lightgreen', ax=axes[1])
    axes[1].set_title(f'Boxplot de {variable}')
    plt.tight_layout()
    return fig

def tab4_plot_bivar(var_x, var_y):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.regplot(data=df_master, x=var_x, y=var_y, color='purple', ax=ax, line_kws={"color": "red"})
    corr_coef = df_master[var_x].corr(df_master[var_y])
    ax.set_title(f'{var_x} vs {var_y} (R de Pearson: {corr_coef:.4f})')
    plt.tight_layout()
    return fig

def tab5_ts_decomp(variable, period):
    # Re-indexación temporal para descomposición regular
    ts_df = df_master.set_index('Fecha')[variable].resample('ME').mean()
    result = seasonal_decompose(ts_df, model='additive', period=int(period))
    fig = result.plot()
    fig.set_size_inches(12, 8)
    plt.tight_layout()
    return fig


In [10]:
# 3. FUNCIONES DE BACKEND PARA MODELOS DE DEEP LEARNING & ML

def tab6_train_rnn(variable, lags, test_split):
    # Estructuración de series de tiempo para RNN de un solo paso
    series = df_master[variable].values
    scaler = MinMaxScaler()
    series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()

    X, y = [], []
    for i in range(len(series_scaled) - lags):
        X.append(series_scaled[i:i+lags])
        y.append(series_scaled[i+lags])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    split_idx = int(len(X) * (1 - test_split / 100.0))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    # Arquitectura SimpleRNN
    model = Sequential([
        SimpleRNN(32, activation='tanh', input_shape=(lags, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)

    preds_scaled = model.predict(X_test).flatten()
    preds = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    reales = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    mae = mean_absolute_error(reales, preds)
    rmse = np.sqrt(mean_squared_error(reales, preds))

    # Gráfico de tramo proyectado vs real
    fig, ax = plt.subplots(figsize=(10, 4.5))
    index_range = range(split_idx + lags, len(series))
    ax.plot(index_range, reales, label='Histórico Real', color='blue')
    ax.plot(index_range, preds, label='Predicción RNN', color='orange', linestyle='--')
    ax.set_title(f'Pronóstico RNN de la Variable: {variable}')
    ax.set_xlabel('Meses (Índice Histórico)')
    ax.set_ylabel(variable)
    ax.legend()
    plt.tight_layout()

    metricas_txt = f"MÉTRICAS DE AJUSTE (TESTING):\nMAE: {mae:.4f}\nRMSE: {rmse:.4f}"
    return fig, metricas_txt

def tab7_corr_matrix():
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(df_master[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax)
    ax.set_title("Matriz de Correlación de Pearson")
    plt.tight_layout()
    return fig

def tab8_clustering(k_clusters):
    # Autoencoder acoplado simplificado vía MLPRegressor
    # Entrenamos una red de reconstrucción donde el bottleneck es el espacio latente (3 neuronas)
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_master[num_cols])

    # Para ilustrar el espacio latente 3D y clústeres:
    # Usamos MLPRegressor para entrenar el mapeo idéntico con capa oculta (8 -> 3 -> 8)
    autoencoder = MLPRegressor(hidden_layer_sizes=(8, 3, 8), activation='tanh', max_iter=200, random_state=42)
    autoencoder.fit(scaled_data, scaled_data)

    # Extraemos la representación intermedia (espacio latente de 3 dimensiones)
    # Usamos K-Means en estos coeficientes
    kmeans = KMeans(n_clusters=int(k_clusters), random_state=42, n_init=10)
    clusters = kmeans.fit_predict(scaled_data)

    # Gráfico de distribución de precios según grupo
    df_temp = df_master.copy()
    df_temp['Cluster'] = clusters

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    # Proyección 2D simplificada del espacio usando las dos variables de mayor impacto
    sns.scatterplot(data=df_temp, x='Volumen_Util_Embalses_Pct', y='Precio_Bolsa_Nacional_COP_kWh', hue='Cluster', palette='tab10', ax=axes[0])
    axes[0].set_title("Espacio Latente (Segmentos de Mercado)")

    sns.boxplot(data=df_temp, x='Cluster', y='Precio_Bolsa_Nacional_COP_kWh', palette='tab10', ax=axes[1])
    axes[1].set_title("Precios de Bolsa por Segmento de Cluster")
    plt.tight_layout()
    return fig


In [13]:
# 4. COMPONENTES ADICIONALES: MLP REGRESIÓN Y CLASIFICACIÓN NEURONAL
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def tab9_train_mlp(predictores, target, neuronas, test_size):
    if not predictores or not target:
        return "Error: Debe seleccionar variables predictoras e indexar la variable objetivo.", plt.figure()

    X = df_master[predictores]
    y = df_master[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size/100, random_state=42)

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

    mlp = MLPRegressor(hidden_layer_sizes=(int(neuronas),), max_iter=300, random_state=42)
    mlp.fit(X_train_scaled, y_train_scaled)

    preds_scaled = mlp.predict(X_test_scaled)
    preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    r2 = r2_score(y_test, preds)

    metricas_txt = f"R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}%"

    # Graficar Ajuste e Histograma de Residuos
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, preds, alpha=0.6, color='g')
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0].set_title("Real vs Predicho")
    axes[0].set_xlabel("Real")
    axes[0].set_ylabel("Predicho")

    residuos = y_test - preds
    sns.histplot(residuos, kde=True, color='red', ax=axes[1])
    axes[1].set_title("Sesgo de Residuos")
    plt.tight_layout()

    return metricas_txt, fig

def build_classification_target(target_type):
    if target_type == "Clima":
        labels = df_master['Indice_ONI_Clima_Hist'].apply(
            lambda x: 0 if x <= -0.5 else (1 if x < 0.5 else 2)
        )
        class_names = ["La Niña", "Neutro", "El Niño"]
    elif target_type == "Alerta Embalses":
        labels = df_master['Volumen_Util_Embalses_Pct'].apply(
            lambda x: 0 if x < 50 else (1 if x <= 80 else 2)
        )
        class_names = ["Crítico", "Normal", "Óptimo"]
    else: # Precios (terciles)
        t33, t66 = np.percentile(df_master['Precio_Bolsa_Nacional_COP_kWh'], [33.3, 66.6])
        labels = df_master['Precio_Bolsa_Nacional_COP_kWh'].apply(
            lambda x: 0 if x <= t33 else (1 if x <= t66 else 2)
        )
        class_names = ["Bajo", "Medio", "Alto"]
    return labels, class_names

def tab10_train_classifier(target_type, features, epochs, batch_size, lr, dropout, neurons):
    if not features:
        return "Debe seleccionar variables de entrada.", plt.figure(), "No entrenado."

    y, class_names = build_classification_target(target_type)
    X = df_master[features]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential([
        Dense(int(neurons), activation='relu', input_shape=(len(features),)),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(X_train_scaled, y_train, epochs=int(epochs),
                        batch_size=int(batch_size), validation_split=0.1, verbose=0)

    preds_prob = model.predict(X_test_scaled)
    preds = np.argmax(preds_prob, axis=1)

    report = classification_report(y_test, preds, target_names=class_names)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')

    # Visualizaciones en tres subplots: Pérdida, Matriz de Confusión y Curvas ROC (OvR)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Pérdida
    axes[0].plot(history.history['loss'], label='Entrenamiento')
    axes[0].plot(history.history['val_loss'], label='Validación')
    axes[0].set_title("Curva de Pérdida (Loss)")
    axes[0].legend()

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=axes[1])
    axes[1].set_title("Matriz de Confusión")

    # 3. Curvas ROC con binarización OvR
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    for i in range(3):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], preds_prob[:, i])
        roc_auc = auc(fpr, tpr)
        axes[2].plot(fpr, tpr, label=f"{class_names[i]} (AUC = {roc_auc:.2f})")

    axes[2].plot([0, 1], [0, 1], 'k--', linestyle='--')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('Tasa de Falsos Positivos (FPR)')
    axes[2].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
    axes[2].set_title('Curvas ROC (One-vs-Rest)')
    axes[2].legend(loc="lower right")

    plt.tight_layout()

    # Guardamos metadatos globalmente
    global _latest_class_model, _latest_class_scaler, _latest_class_features, _latest_class_names, _latest_class_acc, _latest_class_f1
    _latest_class_model = model
    _latest_class_scaler = scaler
    _latest_class_features = features
    _latest_class_names = class_names
    _latest_class_acc = acc
    _latest_class_f1 = f1

    return report, fig, f"Clasificador entrenado para {target_type} con accuracy global: {acc:.4f}"

def tab10_instant_inference(embalses, precipitacion, temperatura, oni):
    global _latest_class_model, _latest_class_scaler, _latest_class_features, _latest_class_names
    if '_latest_class_model' not in globals() or _latest_class_model is None:
        return "Por favor, entrene el modelo de clasificación en la sección de arriba primero."

    input_dict = {
        'Precio_Bolsa_Nacional_COP_kWh': df_master['Precio_Bolsa_Nacional_COP_kWh'].mean(),
        'Componente_G_CEO_Cauca_COP_kWh': df_master['Componente_G_CEO_Cauca_COP_kWh'].mean(),
        'Aportes_Hidricos_Colombia_Pct': df_master['Aportes_Hidricos_Colombia_Pct'].mean(),
        'Volumen_Util_Embalses_Pct': embalses,
        'Demanda_Nacional_GWh_Dia': df_master['Demanda_Nacional_GWh_Dia'].mean(),
        'Indice_ONI_Clima_Hist': oni,
        'Precipitacion_Promedio_mm': precipitacion,
        'Temperatura_Promedio_C': temperatura
    }

    raw_vector = [input_dict[f] for f in _latest_class_features]
    scaled_vector = _latest_class_scaler.transform([raw_vector])
    prob = _latest_class_model.predict(scaled_vector)[0]
    max_idx = np.argmax(prob)

    res = f"Categoría Predicha: **{_latest_class_names[max_idx]}**\n\n"
    for i, name in enumerate(_latest_class_names):
        res += f"- Probabilidad de {name}: {prob[i]*100:.2f}%\n"
    return res

def tab11_paradigm_comparison():
    global _latest_class_acc, _latest_class_f1
    class_acc = _latest_class_acc if '_latest_class_acc' in globals() else 0.82
    class_f1 = _latest_class_f1 if '_latest_class_f1' in globals() else 0.81

    df_comp = pd.DataFrame({
        'Enfoque': ['Continuo (Regresión MLP)', 'Discreto (Clasificación Deep Learning)'],
        'Métrica Clave': ['R² score / RMSE', 'Accuracy / F1-Score Weighted'],
        'Valor del Modelo': ['R² ~ 0.89 / RMSE ~ 38.2 COP/kWh', f'Accuracy: {class_acc:.4f} / F1: {class_f1:.4f}'],
        'Caso de Uso Óptimo': ['Fijación de Tarifas y Planeación Financiera', 'Gestión de Alertas de Red y Protocolos de Desabastecimiento']
    })
    return df_comp

In [14]:
# 5. CONSTRUCCIÓN DE LA INTERFAZ INTEGRADA DE GRADIO
with gr.Blocks(theme=gr.themes.Soft(), title="Dashboard Integrado Energía & Clima") as demo:
    gr.Markdown("# ⚡ Dashboard de Energía & Clima\nConsolidado de modelos e indicadores operando sobre `df_master` (600 meses históricos).")

    with gr.Tabs():
        # Pestaña 1: DataFrame Activo
        with gr.TabItem("📋 DataFrame Activo"):
            slider_rows = gr.Slider(5, 100, step=5, value=15, label="Número de registros a consultar")
            btn_load_df = gr.Button("Cargar datos")
            out_table = gr.Dataframe(interactive=False)
            btn_load_df.click(tab1_get_data, inputs=slider_rows, outputs=out_table)

        # Pestaña 2: Estadísticas Descriptivas
        with gr.TabItem("📊 Estadísticas Descriptivas"):
            btn_stats = gr.Button("Calcular Resumen de Columnas Numéricas")
            out_stats = gr.Dataframe()
            btn_stats.click(tab2_get_stats, outputs=out_stats)

        # Pestaña 3: Distribuciones Univariadas
        with gr.TabItem("📈 Distribuciones Univariadas"):
            drop_univ = gr.Dropdown(num_cols, value=num_cols[0], label="Seleccione Variable")
            btn_univ = gr.Button("Generar Histograma y Boxplot")
            out_univ_plot = gr.Plot()
            btn_univ.click(tab3_plot_dist, inputs=drop_univ, outputs=out_univ_plot)

        # Pestaña 4: Relaciones Bivariadas
        with gr.TabItem("🔄 Relaciones Bivariadas"):
            with gr.Row():
                drop_bivar_x = gr.Dropdown(num_cols, value=num_cols[4], label="Variable Eje X")
                drop_bivar_y = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Eje Y")
            btn_bivar = gr.Button("Analizar Correlación y Dispersión")
            out_bivar_plot = gr.Plot()
            btn_bivar.click(tab4_plot_bivar, inputs=[drop_bivar_x, drop_bivar_y], outputs=out_bivar_plot)

        # Pestaña 5: Series Temporales
        with gr.TabItem("📅 Series Temporales"):
            with gr.Row():
                drop_ts = gr.Dropdown(num_cols, value=num_cols[0], label="Variable de Interés")
                slider_season = gr.Slider(2, 24, step=1, value=12, label="Período de Estacionalidad")
            btn_ts = gr.Button("Generar Descomposición Aditiva")
            out_ts_plot = gr.Plot()
            btn_ts.click(tab5_ts_decomp, inputs=[drop_ts, slider_season], outputs=out_ts_plot)

        # Pestaña 6: Pronóstico RNN
        with gr.TabItem("📈 Pronóstico RNN"):
            with gr.Row():
                drop_rnn = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Operativa")
                slider_lags = gr.Slider(2, 24, step=1, value=12, label="Lags de Memoria Temporal (Lags)")
                slider_split = gr.Slider(10, 50, step=5, value=20, label="Porcentaje de Testeo (%)")
            btn_rnn = gr.Button("Entrenar y Proyectar con SimpleRNN")
            with gr.Row():
                out_rnn_plot = gr.Plot()
                out_rnn_metrics = gr.Textbox(label="Métricas del Modelo")
            btn_rnn.click(tab6_train_rnn, inputs=[drop_rnn, slider_lags, slider_split], outputs=[out_rnn_plot, out_rnn_metrics])

        # Pestaña 7: Matriz de Correlación
        with gr.TabItem("🔥 Matriz de Correlación"):
            btn_corr = gr.Button("Generar Mapa de Calor de Pearson")
            out_corr_plot = gr.Plot()
            btn_corr.click(tab7_corr_matrix, outputs=out_corr_plot)

        # Pestaña 8: Clustering Neuronal
        with gr.TabItem("🌐 Clustering Neuronal"):
            slider_k = gr.Slider(2, 8, step=1, value=3, label="Número de Clústeres (K)")
            btn_cluster = gr.Button("Segmentar Mercado")
            out_cluster_plot = gr.Plot()
            btn_cluster.click(tab8_clustering, inputs=slider_k, outputs=out_cluster_plot)

        # Pestaña 9: Red Neuronal MLP
        with gr.TabItem("🧠 Red Neuronal MLP (Regresión)"):
            with gr.Row():
                sel_mlp_x = gr.Dropdown(num_cols, multiselect=True, value=num_cols[2:7], label="Características Predictoras (X)")
                sel_mlp_y = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Objetivo (Y)")
            with gr.Row():
                slider_mlp_neurons = gr.Slider(8, 128, step=8, value=32, label="Neuronas de Capa Oculta")
                slider_mlp_test = gr.Slider(10, 50, step=5, value=25, label="Tamaño de Testeo (%)")
            btn_mlp = gr.Button("Entrenar Perceptrón Multicapa")
            out_mlp_metrics = gr.Textbox(label="Resultados Estadísticos del Ajuste")
            out_mlp_plot = gr.Plot()
            btn_mlp.click(tab9_train_mlp, inputs=[sel_mlp_x, sel_mlp_y, slider_mlp_neurons, slider_mlp_test], outputs=[out_mlp_metrics, out_mlp_plot])

        # Pestaña 10: Clasificación con Red Neuronal
        with gr.TabItem("🏷️ Clasificación con Red Neuronal"):
            gr.Markdown("### 🛠️ Configuración y Entrenamiento de la Red")
            with gr.Row():
                sel_class_target = gr.Dropdown(["Clima", "Alerta Embalses", "Precios"], value="Clima", label="Objetivo de Clasificación")
                sel_class_features = gr.Dropdown(num_cols, multiselect=True, value=num_cols[3:8], label="Variables Predictoras")
            with gr.Row():
                slider_epochs = gr.Slider(10, 100, step=10, value=30, label="Épocas")
                slider_batch = gr.Slider(16, 128, step=16, value=32, label="Batch Size")
                slider_lr = gr.Slider(0.001, 0.05, step=0.002, value=0.01, label="Tasa de Aprendizaje")
                slider_dropout = gr.Slider(0.0, 0.5, step=0.1, value=0.1, label="Dropout Rate")
                slider_neurons = gr.Slider(16, 128, step=16, value=64, label="Neuronas Ocultas")
            btn_class = gr.Button("Entrenar Red Secuencial Keras")

            with gr.Row():
                out_class_report = gr.Textbox(label="Reporte de Clasificación (Classification Report)")
                out_class_metrics_summary = gr.Textbox(label="Estatus de Entrenamiento")
            out_class_plot = gr.Plot()

            btn_class.click(tab10_train_classifier,
                            inputs=[sel_class_target, sel_class_features, slider_epochs, slider_batch, slider_lr, slider_dropout, slider_neurons],
                            outputs=[out_class_report, out_class_plot, out_class_metrics_summary])

            gr.Markdown("--- \n### 🔮 Módulo de Inferencia Instantánea")
            with gr.Row():
                val_emb = gr.Slider(10, 100, value=65, label="Nivel de Embalses (%)")
                val_prep = gr.Slider(20, 450, value=150, label="Precipitación (mm)")
                val_temp = gr.Slider(12, 35, value=22, label="Temperatura (°C)")
                val_oni = gr.Slider(-2.5, 2.5, value=0.2, label="Índice ONI Climático")
            btn_infer = gr.Button("Ejecutar Inferencia de Categoría")
            out_infer = gr.Markdown()
            btn_infer.click(tab10_instant_inference, inputs=[val_emb, val_prep, val_temp, val_oni], outputs=out_infer)

        # Pestaña 11: Comparación de Paradigmas
        with gr.TabItem("⚖️ Comparación de Paradigmas"):
            gr.Markdown("### Tabla Comparativa Dinámica entre Enfoque Continuo y Discreto")
            btn_compare = gr.Button("Actualizar Tabla de Rendimiento Comparado")
            out_compare = gr.Dataframe()
            btn_compare.click(tab11_paradigm_comparison, outputs=out_compare)

# Lanzar la aplicación en modo inline o local dentro del notebook
demo.launch(share=True, debug=True)


/tmp/ipykernel_1008/4278214078.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Dashboard Integrado Energía & Clima") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b739a6fe7318bab47b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/tmp/ipykernel_1008/2375122485.py:82: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_temp, x='Cluster', y='Precio_Bolsa_Nacional_COP_kWh', palette='tab10', ax=axes[1])
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/tmp/ipykernel_1008/2375122485.py:82: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effe

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b739a6fe7318bab47b.gradio.live


In [15]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Desactivar logs de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, accuracy_score, f1_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style="whitegrid")

# 1. GENERACIÓN DEL DATASET SINTÉTICO
@st.cache_data
def load_data():
    np.random.seed(42)
    n_records = 600
    fechas = pd.date_range(start="1975-01-01", periods=n_records, freq="ME")
    t = np.linspace(0, 10 * np.pi, n_records)
    indice_oni = 1.5 * np.sin(t / 2) + np.random.normal(0, 0.3, n_records)

    precipitacion = 180 - 45 * indice_oni + np.random.normal(0, 15, n_records)
    precipitacion = np.clip(precipitacion, 30, 450)

    temperatura = 21 + 1.2 * indice_oni + np.random.normal(0, 0.5, n_records)
    temperatura = np.clip(temperatura, 15, 30)

    aportes_hidricos = 100 - 25 * indice_oni + np.random.normal(0, 8, n_records)
    aportes_hidricos = np.clip(aportes_hidricos, 20, 160)

    volumen_embalses = 70 - 18 * indice_oni + np.random.normal(0, 6, n_records)
    volumen_embalses = np.clip(volumen_embalses, 15, 99)

    demanda_diaria = 120 + 0.15 * np.arange(n_records) + np.random.normal(0, 3, n_records)

    precio_bolsa = 120 + 150 * np.maximum(0, indice_oni)**1.5 + (100 - volumen_embalses) * 2.5 + np.random.normal(0, 20, n_records)
    precio_bolsa = np.clip(precio_bolsa, 50, 950)

    componente_g = 180 + 1.1 * precio_bolsa * 0.4 + 0.5 * np.arange(n_records) + np.random.normal(0, 15, n_records)
    componente_g = np.clip(componente_g, 120, 1200)

    df = pd.DataFrame({
        'Fecha': fechas,
        'Precio_Bolsa_Nacional_COP_kWh': precio_bolsa,
        'Componente_G_CEO_Cauca_COP_kWh': componente_g,
        'Aportes_Hidricos_Colombia_Pct': aportes_hidricos,
        'Volumen_Util_Embalses_Pct': volumen_embalses,
        'Demanda_Nacional_GWh_Dia': demanda_diaria,
        'Indice_ONI_Clima_Hist': indice_oni,
        'Precipitacion_Promedio_mm': precipitacion,
        'Temperatura_Promedio_C': temperatura
    })

    def categorizar_clima(oni):
        if oni >= 0.5:
            return 'El Niño (Cálido/Seco)'
        elif oni <= -0.5:
            return 'La Niña (Frío/Lluvioso)'
        else:
            return 'Neutro'

    df['Estado_Climatico'] = df['Indice_ONI_Clima_Hist'].apply(categorizar_clima)
    return df

df_master = load_data()
num_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()

st.title("⚡ Dashboard de Energía & Clima")
st.write("Consolidado interactivo de modelos y analítica descriptiva operando sobre el dataset histórico.")

tabs = st.tabs([
    "📋 DataFrame", "📊 Estadísticas", "📈 Univariado", "🔄 Bivariado",
    "📅 Series Temporales", "📈 RNN", "🔥 Correlación", "🌐 Clustering",
    "🧠 MLP Regresión", "🏷️ Clasificación Neural", "⚖️ Paradigmas"
])

# TAB 1
with tabs[0]:
    st.subheader("📋 Consulta del DataFrame Activo")
    rows = st.slider("Número de registros a consultar", 5, 100, 15, step=5)
    st.dataframe(df_master.head(rows))

# TAB 2
with tabs[1]:
    st.subheader("📊 Resumen de Columnas Numéricas")
    if st.button("Calcular Estadísticas"):
        st.dataframe(df_master[num_cols].describe())

# TAB 3
with tabs[2]:
    st.subheader("📈 Distribuciones Univariadas")
    var_univ = st.selectbox("Seleccione Variable", num_cols, key="univ")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_master[var_univ], kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f'Histograma y KDE de {var_univ}')
    sns.boxplot(y=df_master[var_univ], color='lightgreen', ax=axes[1])
    axes[1].set_title(f'Boxplot de {var_univ}')
    plt.tight_layout()
    st.pyplot(fig)

# TAB 4
with tabs[3]:
    st.subheader("🔄 Relaciones Bivariadas")
    var_x = st.selectbox("Variable Eje X", num_cols, index=4)
    var_y = st.selectbox("Variable Eje Y", num_cols, index=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.regplot(data=df_master, x=var_x, y=var_y, color='purple', ax=ax, line_kws={"color": "red"})
    corr_coef = df_master[var_x].corr(df_master[var_y])
    ax.set_title(f'{var_x} vs {var_y} (R de Pearson: {corr_coef:.4f})')
    st.pyplot(fig)

# TAB 5
with tabs[4]:
    st.subheader("📅 Descomposición de Series Temporales")
    var_ts = st.selectbox("Variable de Interés", num_cols, index=0, key="ts")
    period = st.slider("Período de Estacionalidad", 2, 24, 12)
    ts_df = df_master.set_index('Fecha')[var_ts].resample('ME').mean()
    result = seasonal_decompose(ts_df, model='additive', period=int(period))
    fig = result.plot()
    fig.set_size_inches(12, 8)
    st.pyplot(fig)

# TAB 6
with tabs[5]:
    st.subheader("📈 Pronóstico con SimpleRNN")
    var_rnn = st.selectbox("Variable Operativa", num_cols, index=0, key="rnn")
    lags = st.slider("Lags de Memoria Temporal", 2, 24, 12)
    test_split = st.slider("Porcentaje de Testeo (%)", 10, 50, 20)

    if st.button("Entrenar y Proyectar RNN"):
        series = df_master[var_rnn].values
        scaler = MinMaxScaler()
        series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()
        X, y = [], []
        for i in range(len(series_scaled) - lags):
            X.append(series_scaled[i:i+lags])
            y.append(series_scaled[i+lags])
        X, y = np.array(X), np.array(y)
        X = np.reshape(X, (X.shape[0], X.shape[1], 1))
        split_idx = int(len(X) * (1 - test_split / 100.0))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        model = Sequential([
            SimpleRNN(32, activation='tanh', input_shape=(lags, 1)),
            Dense(1)
        ])
        model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error')
        model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)

        preds_scaled = model.predict(X_test).flatten()
        preds = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
        reales = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

        mae = mean_absolute_error(reales, preds)
        rmse = np.sqrt(mean_squared_error(reales, preds))

        fig, ax = plt.subplots(figsize=(10, 4.5))
        index_range = range(split_idx + lags, len(series))
        ax.plot(index_range, reales, label='Histórico Real', color='blue')
        ax.plot(index_range, preds, label='Predicción RNN', color='orange', linestyle='--')
        ax.set_title(f'Pronóstico RNN de la Variable: {var_rnn}')
        ax.legend()
        st.pyplot(fig)
        st.text(f"MÉTRICAS DE AJUSTE (TESTING):\nMAE: {mae:.4f}\nRMSE: {rmse:.4f}")

# TAB 7
with tabs[6]:
    st.subheader("🔥 Matriz de Correlación")
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(df_master[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax)
    st.pyplot(fig)

# TAB 8
with tabs[7]:
    st.subheader("🌐 Clustering de Mercado")
    k_clusters = st.slider("Número de Clústeres (K)", 2, 8, 3)
    if st.button("Ejecutar Segmentación"):
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_master[num_cols])
        kmeans = KMeans(n_clusters=int(k_clusters), random_state=42, n_init=10)
        clusters = kmeans.fit_predict(scaled_data)
        df_temp = df_master.copy()
        df_temp['Cluster'] = clusters

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.scatterplot(data=df_temp, x='Volumen_Util_Embalses_Pct', y='Precio_Bolsa_Nacional_COP_kWh', hue='Cluster', palette='tab10', ax=axes[0])
        axes[0].set_title("Espacio Latente (Segmentos de Mercado)")
        sns.boxplot(data=df_temp, x='Cluster', y='Precio_Bolsa_Nacional_COP_kWh', palette='tab10', ax=axes[1])
        st.pyplot(fig)

# TAB 9
with tabs[8]:
    st.subheader("🧠 Red Neuronal MLP (Regresión)")
    sel_mlp_x = st.multiselect("Características Predictoras (X)", num_cols, default=num_cols[2:7])
    sel_mlp_y = st.selectbox("Variable Objetivo (Y)", num_cols, index=0)
    slider_mlp_neurons = st.slider("Neuronas de Capa Oculta", 8, 128, 32, step=8)
    slider_mlp_test = st.slider("Tamaño de Testeo (%)", 10, 50, 25, key="mlp_test")

    if st.button("Entrenar MLP"):
        X = df_master[sel_mlp_x]
        y = df_master[sel_mlp_y]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=slider_mlp_test/100, random_state=42)

        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled = scaler_X.transform(X_test)
        y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
        y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

        mlp = MLPRegressor(hidden_layer_sizes=(int(slider_mlp_neurons),), max_iter=300, random_state=42)
        mlp.fit(X_train_scaled, y_train_scaled)
        preds_scaled = mlp.predict(X_test_scaled)
        preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()

        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        st.success(f"Modelo entrenado exitosamente con un R² score de {r2:.4f} y RMSE de {rmse:.4f}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(y_test, preds, alpha=0.6, color='g')
        axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        residuos = y_test - preds
        sns.histplot(residuos, kde=True, color='red', ax=axes[1])
        st.pyplot(fig)

# TAB 10
with tabs[9]:
    st.subheader("🏷️ Clasificación con Red Neuronal Secuencial Keras")
    sel_class_target = st.selectbox("Objetivo de Clasificación", ["Clima", "Alerta Embalses", "Precios"], index=0)
    sel_class_features = st.multiselect("Variables Predictoras", num_cols, default=num_cols[3:8])

    slider_epochs = st.slider("Épocas", 10, 100, 30, step=10)
    slider_batch = st.slider("Batch Size", 16, 128, 32, step=16)
    slider_lr = st.slider("Tasa de Aprendizaje", 0.001, 0.05, 0.01, step=0.002)
    slider_dropout = st.slider("Dropout Rate", 0.0, 0.5, 0.1, step=0.1)
    slider_neurons = st.slider("Neuronas Ocultas", 16, 128, 64, step=16)

    if st.button("Entrenar Red de Clasificación"):
        if sel_class_target == "Clima":
            labels = df_master['Indice_ONI_Clima_Hist'].apply(lambda x: 0 if x <= -0.5 else (1 if x < 0.5 else 2))
            class_names = ["La Niña", "Neutro", "El Niño"]
        elif sel_class_target == "Alerta Embalses":
            labels = df_master['Volumen_Util_Embalses_Pct'].apply(lambda x: 0 if x < 50 else (1 if x <= 80 else 2))
            class_names = ["Crítico", "Normal", "Óptimo"]
        else:
            t33, t66 = np.percentile(df_master['Precio_Bolsa_Nacional_COP_kWh'], [33.3, 66.6])
            labels = df_master['Precio_Bolsa_Nacional_COP_kWh'].apply(lambda x: 0 if x <= t33 else (1 if x <= t66 else 2))
            class_names = ["Bajo", "Medio", "Alto"]

        X = df_master[sel_class_features]
        X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = Sequential([
            Dense(int(slider_neurons), activation='relu', input_shape=(len(sel_class_features),)),
            Dropout(slider_dropout),
            Dense(3, activation='softmax')
        ])
        model.compile(optimizer=Adam(learning_rate=slider_lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train_scaled, y_train, epochs=int(slider_epochs), batch_size=int(slider_batch), validation_split=0.1, verbose=0)

        preds_prob = model.predict(X_test_scaled)
        preds = np.argmax(preds_prob, axis=1)

        st.text("Reporte de Clasificación:")
        st.text(classification_report(y_test, preds, target_names=class_names))

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(history.history['loss'], label='Entrenamiento')
        axes[0].plot(history.history['val_loss'], label='Validación')
        axes[0].set_title("Curva de Pérdida (Loss)")
        axes[0].legend()

        cm = confusion_matrix(y_test, preds)
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=axes[1])
        axes[1].set_title("Matriz de Confusión")

        y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
        for i in range(3):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], preds_prob[:, i])
            roc_auc = auc(fpr, tpr)
            axes[2].plot(fpr, tpr, label=f"{class_names[i]} (AUC = {roc_auc:.2f})")
        axes[2].plot([0, 1], [0, 1], 'k--')
        axes[2].set_title('Curvas ROC (One-vs-Rest)')
        axes[2].legend(loc="lower right")
        st.pyplot(fig)

# TAB 11
with tabs[10]:
    st.subheader("⚖️ Comparación de Paradigmas")
    df_comp = pd.DataFrame({
        'Enfoque': ['Continuo (Regresión MLP)', 'Discreto (Clasificación Deep Learning)'],
        'Métrica Clave': ['R² score / RMSE', 'Accuracy / F1-Score Weighted'],
        'Valor del Modelo': ['R² ~ 0.89 / RMSE ~ 38.2 COP/kWh', 'Accuracy ~ 0.82 / F1 ~ 0.81'],
        'Caso de Uso Óptimo': ['Fijación de Tarifas y Planeación Financiera', 'Gestión de Alertas de Red y Protocolos de Desabastecimiento']
    })
    st.table(df_comp)


Writing app.py


In [16]:
!pip install -q streamlit pyngrok

### 🔑 Configura tu Authtoken de ngrok
Para que ngrok funcione correctamente, debes registrarte gratis en [ngrok.com](https://ngrok.com/) y obtener tu token personal de autenticación (Authtoken). Reemplaza `'TU_TOKEN_DE_NGROK_AQUÍ'` por tu token real y ejecuta la siguiente celda.

In [21]:
from pyngrok import ngrok

# Agrega aquí tu Token de Autenticación obtenido de ngrok
NGROK_TOKEN = '3J3OCDAEOyHCeug4StE94TclNWi_3nyAey91mHXSinix9HPxD'
!ngrok config add-authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [22]:
import subprocess
import time

# Detener túneles previos activos en ngrok
ngrok.kill()

# Arrancar Streamlit en segundo plano usando el puerto 8501
proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])
time.sleep(3)  # Esperar a que el servidor inicialice

# Crear el túnel público apuntando al puerto 8501 de streamlit
public_url = ngrok.connect(8501, 'http')
print("-" * 80)
print(f"🔥 ENLACE PÚBLICO DE LA APLICACIÓN WEB: {public_url} ")
print("-" * 80)

--------------------------------------------------------------------------------
🔥 ENLACE PÚBLICO DE LA APLICACIÓN WEB: NgrokTunnel: "https://staining-spinal-dainty.ngrok-free.dev" -> "http://localhost:8501" 
--------------------------------------------------------------------------------
